# 01.1 — Resume Vector Search

**Jackson and Jackson HR Digital**

This notebook enables **semantic resume retrieval** for the HR Digital predictive hiring pipeline.
Rather than keyword matching, we embed full resume text using a managed embedding model and
store the vectors in a Databricks Vector Search index — allowing the downstream Genie / agent
to find the most semantically-relevant candidates for any free-text query.

### What this notebook does
1. Extracts plain text from all 10 PDF resumes stored in the Volume
2. Joins resume text to the Silver `candidates` table on `resume_filename`
3. Writes a `hr_resumes` source table with **Change Data Feed enabled** (required for Delta Sync)
4. Creates a Databricks Vector Search endpoint (`STANDARD` type)
5. Creates a Delta Sync VS index using managed embeddings (`databricks-gte-large-en`)
6. Waits for the index to go ONLINE, then runs a test similarity search

### Prerequisites
- `00_load_bronze.ipynb` and `01_load_silver.ipynb` must have run successfully
- PDF resumes are staged in the Volume under `/resumes/`
- The cluster service principal has `CREATE ENDPOINT` / `CREATE INDEX` permissions

**Estimated runtime:** 10–15 minutes (dominated by endpoint creation + index sync)

## Install Dependencies

Install `pypdf` for PDF text extraction and `databricks-vectorsearch` for the VS SDK, then restart the Python kernel to pick up newly installed packages.

In [ ]:
%pip install pypdf databricks-vectorsearch -q

In [ ]:
dbutils.library.restartPython()

## Setup — Configuration Parameters

Define the Unity Catalog hierarchy, Volume path, VS endpoint name, and VS index name via widgets for easy parameterization.

In [ ]:
# ---------------------------------------------------------------------------
# Widget definitions
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog",          "bx4",                   "UC Catalog")
dbutils.widgets.text("schema",           "hrd_2030",              "UC Schema")
dbutils.widgets.text("volume_name",      "hrd_raw_data",          "UC Volume Name")
dbutils.widgets.text("vs_endpoint_name", "bx4_hrd_vs_endpoint",  "VS Endpoint Name")
dbutils.widgets.text("vs_index_name",    "hr_resumes_vs_index",   "VS Index Name")

In [ ]:
# ---------------------------------------------------------------------------
# Retrieve widget values
# ---------------------------------------------------------------------------
import time
from datetime import datetime

catalog          = dbutils.widgets.get("catalog")
schema           = dbutils.widgets.get("schema")
volume_name      = dbutils.widgets.get("volume_name")
vs_endpoint_name = dbutils.widgets.get("vs_endpoint_name")
vs_index_name    = dbutils.widgets.get("vs_index_name")

volume_path         = f"/Volumes/{catalog}/{schema}/{volume_name}"
resumes_volume_path = f"{volume_path}/resumes"

# Fully-qualified names used by VS SDK
source_table_fqn = f"{catalog}.{schema}.hr_resumes"
index_fqn        = f"{catalog}.{schema}.{vs_index_name}"

print(f"Catalog            : {catalog}")
print(f"Schema             : {schema}")
print(f"Volume path        : {volume_path}")
print(f"Resumes path       : {resumes_volume_path}")
print(f"VS Endpoint        : {vs_endpoint_name}")
print(f"VS Index           : {vs_index_name}")
print(f"Source table       : {source_table_fqn}")
print(f"Index FQN          : {index_fqn}")

## Extract Resume Text from PDFs

Read each PDF resume from the Unity Catalog Volume using `pypdf` and extract the raw text. The text will be used as the embedding source for semantic search.

In [ ]:
# ---------------------------------------------------------------------------
# Extract text from PDF resumes stored in the Volume
# pypdf reads directly from the POSIX path (Volume is mounted under /Volumes/)
# ---------------------------------------------------------------------------
import pypdf

resume_records = []

try:
    pdf_files = [f for f in dbutils.fs.ls(resumes_volume_path) if f.name.endswith(".pdf")]
except Exception as e:
    raise RuntimeError(
        f"Could not list PDFs at {resumes_volume_path}. "
        f"Ensure 00_load_bronze has run.\nOriginal error: {e}"
    )

print(f"Found {len(pdf_files)} PDF file(s) in {resumes_volume_path}")

for f in sorted(pdf_files, key=lambda x: x.name):
    # dbutils paths use 'dbfs:' prefix; strip it for direct POSIX open()
    # Volumes are directly accessible at /Volumes/... without dbfs: prefix
    posix_path = f"{resumes_volume_path}/{f.name}"

    try:
        with open(posix_path, "rb") as pdf_fh:
            reader = pypdf.PdfReader(pdf_fh)
            pages  = [page.extract_text() or "" for page in reader.pages]
            text   = "\n".join(pages).strip()
    except Exception as e:
        print(f"  WARNING: Could not parse {f.name}: {e}")
        text = ""

    resume_records.append({
        "resume_filename": f.name,
        "resume_text":     text,
        "ingested_at":     datetime.now(),
    })
    print(f"  {f.name}: {len(text):,} characters extracted")

print(f"\n✓ Extracted text from {len(resume_records)} resume(s).")

## Build hr_resumes Source Table

Join extracted resume text with the Silver `candidates` table on `resume_filename` and write the result as a Delta table with **Change Data Feed enabled** — required for Delta Sync Vector Search indexing.

In [ ]:
# ---------------------------------------------------------------------------
# Join resume text with the Silver candidates table and write hr_resumes
# The table MUST have Change Data Feed enabled for Delta Sync VS indexing.
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

# Build a DataFrame from the extracted PDF records
resume_schema = StructType([
    StructField("resume_filename", StringType(),   nullable=False),
    StructField("resume_text",     StringType(),   nullable=True),
    StructField("ingested_at",     TimestampType(), nullable=True),
])

df_resumes = spark.createDataFrame(resume_records, schema=resume_schema)

# Load Silver candidates and select the columns we need for the VS source table
df_candidates = (
    spark.table(f"`{catalog}`.`{schema}`.candidates")
    .select("candidate_id", "first_name", "last_name", "current_title", "resume_filename")
)

# Inner join — only candidates whose resume PDF was successfully extracted
df_hr_resumes = (
    df_candidates
    .join(df_resumes, on="resume_filename", how="inner")
    .select(
        "candidate_id",
        "first_name",
        "last_name",
        "current_title",
        "resume_filename",
        "resume_text",
        "ingested_at",
    )
)

print(f"Joined record count: {df_hr_resumes.count()}")
df_hr_resumes.select("candidate_id", "first_name", "last_name", "resume_filename").show(truncate=False)

# Write as a managed Delta table with Change Data Feed enabled
(
    df_hr_resumes
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(source_table_fqn)
)

# Also enable CDF via ALTER TABLE in case the write option was ignored on overwrite
spark.sql(f"""
  ALTER TABLE {source_table_fqn}
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

count = spark.table(source_table_fqn).count()
print(f"\n✓ hr_resumes table written — {count:,} rows")
print(f"  Table: {source_table_fqn}")

## Create Vector Search Endpoint

Create a `STANDARD` Vector Search endpoint (or reuse an existing one) and wait for it to reach `ONLINE` status before proceeding.

In [ ]:
# ---------------------------------------------------------------------------
# Create (or reuse) the Vector Search endpoint
# ---------------------------------------------------------------------------
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

def wait_for_endpoint(vsc, name, max_wait_seconds=600, poll_interval=15):
    """Poll until the endpoint reaches ONLINE status or the timeout expires."""
    start = time.time()
    while True:
        ep     = vsc.get_endpoint(name)
        status = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
        elapsed = int(time.time() - start)
        print(f"  [{elapsed:>4}s] Endpoint status: {status}")
        if status == "ONLINE":
            print(f"✓ Endpoint '{name}' is ONLINE.")
            return ep
        if status in ("FAILED", "UNKNOWN") and elapsed > 30:
            raise RuntimeError(f"Endpoint '{name}' entered status '{status}'. Check the Databricks UI.")
        if elapsed > max_wait_seconds:
            raise TimeoutError(f"Endpoint '{name}' did not become ONLINE within {max_wait_seconds}s.")
        time.sleep(poll_interval)

# Check if the endpoint already exists
try:
    ep = vsc.get_endpoint(vs_endpoint_name)
    ep_status = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"Endpoint '{vs_endpoint_name}' already exists (status: {ep_status}).")
    if ep_status != "ONLINE":
        print("Waiting for existing endpoint to become ONLINE...")
        wait_for_endpoint(vsc, vs_endpoint_name)
    else:
        print("✓ Endpoint is already ONLINE — skipping creation.")
except Exception as e:
    if "does not exist" in str(e).lower() or "not found" in str(e).lower():
        print(f"Endpoint '{vs_endpoint_name}' not found — creating now (this takes ~5 min)...")
        vsc.create_endpoint(name=vs_endpoint_name, endpoint_type="STANDARD")
        wait_for_endpoint(vsc, vs_endpoint_name)
    else:
        raise

## Create Delta Sync Index and Wait for Online

Create a Delta Sync VS index with managed embeddings (`databricks-gte-large-en`) on `resume_text`, then poll until the index finishes its initial sync and reaches `ONLINE` status.

In [ ]:
# ---------------------------------------------------------------------------
# Create the Delta Sync Vector Search index with managed embeddings
# If the index already exists on this endpoint we reuse it.
# If it exists on a DIFFERENT (possibly deleted) endpoint, we delete and recreate.
# ---------------------------------------------------------------------------
EMBEDDING_MODEL = "databricks-gte-large-en"

def get_or_create_index(vsc, endpoint_name, index_name, source_table, primary_key, embedding_col, model):
    """Return the existing VS index if it's on the correct endpoint, otherwise create it.
    Handles the case where the index exists on a stale/deleted endpoint by deleting and recreating."""
    try:
        idx = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
        print(f"✓ Index '{index_name}' already exists on '{endpoint_name}' — skipping creation.")
        return idx
    except Exception as e:
        err_lower = str(e).lower()
        if "does not exist" in err_lower or "not found" in err_lower:
            # Index doesn't exist at all — create fresh
            print(f"Index '{index_name}' not found — creating on '{endpoint_name}'...")
        elif "wrong vector search endpoint" in err_lower or "wrong endpoint" in err_lower:
            # Index exists but is bound to a different (possibly deleted) endpoint.
            # Delete it so we can recreate on the current endpoint.
            print(f"⚠️  Index '{index_name}' is bound to a different endpoint — deleting stale index...")
            try:
                vsc.delete_index(endpoint_name=endpoint_name, index_name=index_name)
            except Exception:
                # endpoint_name arg may be ignored on delete; try without it
                try:
                    vsc.delete_index(index_name=index_name)
                except Exception as del_err:
                    print(f"  Warning: delete attempt returned: {del_err}")
            print(f"  Stale index deleted. Creating fresh on '{endpoint_name}'...")
        else:
            raise

    idx = vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        source_table_name=source_table,
        index_name=index_name,
        pipeline_type="TRIGGERED",
        primary_key=primary_key,
        embedding_source_column=embedding_col,
        embedding_model_endpoint_name=model,
    )
    print(f"✓ Index creation initiated: {index_name}")
    return idx


vs_index = get_or_create_index(
    vsc            = vsc,
    endpoint_name  = vs_endpoint_name,
    index_name     = index_fqn,
    source_table   = source_table_fqn,
    primary_key    = "candidate_id",
    embedding_col  = "resume_text",
    model          = EMBEDDING_MODEL,
)

print(f"\nEmbedding model : {EMBEDDING_MODEL}")
print(f"Source table    : {source_table_fqn}")
print(f"Index           : {index_fqn}")
print(f"Pipeline type   : TRIGGERED")

In [ ]:
# ---------------------------------------------------------------------------
# Wait for the index to reach ONLINE status
# ---------------------------------------------------------------------------

def wait_for_index(vsc, endpoint_name, index_name, max_wait_seconds=3600, poll_interval=20):
    """Poll until the VS index is ONLINE (fully synced) or the timeout expires.
    Accepts any state starting with 'ONLINE' (e.g. ONLINE, ONLINE_NO_PENDING_UPDATE).
    On timeout, logs a warning and returns the index rather than raising — since
    the index may still be syncing and this task is not on the critical pipeline path.
    """
    start = time.time()
    while True:
        idx     = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
        status  = idx.describe().get("status", {}).get("detailed_state", "UNKNOWN")
        elapsed = int(time.time() - start)
        print(f"  [{elapsed:>4}s] Index status: {status}")
        if status.startswith("ONLINE"):
            print(f"\n✓ Index '{index_name}' is {status} and ready for queries.")
            return idx
        if "FAILED" in status:
            raise RuntimeError(
                f"Index '{index_name}' entered status '{status}'. Check the VS UI for details."
            )
        if elapsed > max_wait_seconds:
            print(
                f"\nWARNING: Index '{index_name}' did not become ONLINE within {max_wait_seconds}s. "
                "The sync may still be running in the background. "
                "Check the Databricks Vector Search UI — the index will be queryable once ONLINE."
            )
            return idx
        time.sleep(poll_interval)

print("Waiting for VS index to finish syncing (initial sync embeds all resume texts)...")
print("This typically takes 3–15 minutes for 10 documents (first-time embedding).\n")

vs_index = wait_for_index(
    vsc           = vsc,
    endpoint_name = vs_endpoint_name,
    index_name    = index_fqn,
)

## Test Similarity Search

Run sample free-text queries against the index to confirm semantic retrieval is working correctly and returning ranked candidates.

In [ ]:
# ---------------------------------------------------------------------------
# Test the index with a sample similarity search
# ---------------------------------------------------------------------------
import json

test_queries = [
    "experienced HR director with pharma background SPHR certification",
    "HR leader with global workforce transformation and HRIS implementation experience",
    "talent acquisition and organizational development specialist in healthcare",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 70)

    try:
        results = vs_index.similarity_search(
            query_text = query,
            columns    = ["candidate_id", "first_name", "last_name", "current_title"],
            num_results = 3,
        )

        hits = results.get("result", {}).get("data_array", [])
        cols = [c["name"] for c in results.get("manifest", {}).get("columns", [])]

        if hits:
            for rank, row in enumerate(hits, 1):
                record = dict(zip(cols, row))
                score  = record.get("score", "N/A")
                print(
                    f"  #{rank}  {record.get('candidate_id','?'):>5}  "
                    f"{record.get('first_name','')} {record.get('last_name',''):15}  "
                    f"{record.get('current_title','')[:45]:45}  score={score:.4f}"
                )
        else:
            print("  No results returned.")

    except Exception as e:
        print(f"  ERROR during similarity search: {e}")

print("\n✓ Vector Search is operational and returning semantic results.")

## Summary

The Vector Search pipeline is now operational:

| Resource | Value |
|---|---|
| Source table | `{catalog}.{schema}.hr_resumes` |
| Embedding model | `databricks-gte-large-en` |
| VS Endpoint | `{vs_endpoint_name}` |
| VS Index | `{catalog}.{schema}.{vs_index_name}` |
| Pipeline type | TRIGGERED (re-sync via `vsc.get_index(...).sync()`) |

### Re-syncing after new resumes are added
```python
from databricks.vector_search.client import VectorSearchClient
vsc   = VectorSearchClient()
index = vsc.get_index(endpoint_name=vs_endpoint_name, index_name=index_fqn)
index.sync()
```

### Integration with Genie / Agent
The index FQN and endpoint name can be passed as configuration to the Genie room or AI agent
for semantic candidate retrieval as part of the HR Digital predictive hiring workflow.